← [Índice](00-indice.ipynb) · Siguiente: [Dos formas de saber](02-dos-formas-de-saber.ipynb) →

# 01 · El problema



## Estado vigente del proyecto (actualizado el 13 de agosto de 2026)

Esta serie conserva explicaciones y resultados históricos, pero la referencia
operativa actual es la siguiente:

- El sistema experto tiene **193 reglas**, CF estilo MYCIN, meta-reglas,
  encadenamiento hacia adelante y hacia atrás. Su voto usa OpenAI como
  proveedor principal, con heurísticas OpenCV que refinan atributos.
- El modelo local que participa en la decisión es **MobileNetV2 TFLite
  float32**, corrida `run_20260721_2129`; clasifica solo `plastico | vidrio`.
  MobileNetV3-Large INT8 está archivado como respaldo y **no emite votos**.
- En 1.000 capturas OV3660/QVGA, V2 obtuvo **71,60 %** de exactitud y
  **71,25 %** de macro-F1; V3 INT8 obtuvo 57,10 % y 57,09 %. La validación
  histórica de 98,43 % no describe por sí sola el rendimiento del robot.
- La ESP32-CAM toma tres fotos: se suman los seis votos válidos de ambas
  fuentes. `desconocido` es abstención; un empate se resuelve con el proveedor.
  Si el proveedor se abstiene las tres veces, el modelo local necesita 3/3.

La documentación operativa es [`ia/vision-service/README.md`](../../ia/vision-service/README.md),
[`model/README.md`](../../ia/vision-service/model/README.md) y
[`PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md`](../../docs/PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md).


## Qué hace Reci

Reci es un robot de reciclaje para el campus de la PUCE Sede Manabí. Se desplaza
entre puntos fijos, alguien le acerca un residuo, y el robot debe abrir **una de
dos compuertas**: vidrio o plástico.

Toda la inteligencia artificial del proyecto existe para responder esa única
pregunta, y tiene tres respuestas posibles:

```
vidrio       → abre la compuerta de vidrio
plastico     → abre la compuerta de plástico
desconocido  → no abre nada
```

La tercera no es un fracaso: es una decisión de diseño de la que hablaremos en
todos los documentos siguientes.



## El recorrido de un residuo

```
persona acerca un residuo
        ↓
ESP32-CAM toma 3 fotos
        ↓
POST /api/vision/classify          (app Next.js, autenticada)
        ↓
Servicio de visión                 (proceso Python aparte)
   cada foto se analiza por dos caminos independientes
        ↓
{ material, confidence, rule_applied }
        ↓
ESP32-CAM → CMD:CLASSIFY:<material> → Arduino Mega → servo
```

Tres fotos por residuo, dos análisis por foto: **seis diagnósticos** para
decidir una sola cosa. El porqué está en
[Votación y decisión](10-votacion-y-decision.ipynb).



## Por qué el problema es difícil

Parece trivial —una botella de vidrio y una de plástico se distinguen a
simple vista— pero para una cámara no lo es.

### 1. El material no siempre se ve

El caso que el proyecto documentó desde el principio: las botellas de Gatorade
existen en vidrio y en plástico, **con la misma marca, la misma forma y la
misma etiqueta**. Cambia el material, no la apariencia. El modelo local falló
en las dos direcciones sobre ese par (`prueba10.jpeg` y `prueba12.jpeg`).

Lo que de verdad distingue vidrio de plástico —peso, dureza, temperatura,
sonido al golpear— no está en una fotografía.

### 2. La cámara es modesta

La ESP32-CAM con sensor **OV3660** captura en **QVGA: 320 × 240 píxeles**, con compresión JPEG,
óptica sencilla y sin control de iluminación. Los reflejos y la transparencia
—las pistas visuales más útiles para el vidrio— son justamente lo primero que
se pierde con poca resolución y luz irregular.

### 3. El campus no es un laboratorio

Fondos cambiantes, luz distinta según la hora, objetos rotados o encuadrados a
medias, botellas aplastadas, etiquetas arrancadas.

### 4. El error tiene costo físico

Si el robot se equivoca, abre la compuerta incorrecta y **contamina un
contenedor entero**. Separar vidrio de plástico ya mezclados es caro. Un error
no es un número que baja en una tabla: es trabajo manual para alguien.

De ahí la política que atraviesa todo el sistema, escrita en el código
(`inference_engine.py`):

> Solo PLASTICO y VIDRIO abren una compuerta física. Ante la duda, se prefiere
> rechazar (DESCONOCIDO) antes que abrir la compuerta equivocada.


## Lo que el sistema no sabe hacer

Vale la pena decirlo temprano, porque condiciona todo lo demás:

- **Solo conoce dos materiales.** Latas, cartón, orgánicos y todo lo demás
  deberían salir como `desconocido`. El sistema experto sí distingue `ORGANICO`
  y `LATA`, pero se colapsan a `desconocido` porque el robot solo tiene dos
  compuertas.
- **La red neuronal no puede abstenerse.** Es un clasificador binario: siempre
  reparte su probabilidad entre `plastico` y `vidrio`. Si le muestras un mango,
  responderá uno de los dos, a veces con confianza 1,0. La abstención la aporta
  el sistema completo, no el modelo.



## Lo que sigue

El proyecto no resuelve esto con una sola técnica, sino combinando dos familias
de inteligencia artificial que fallan por motivos distintos. Esa es la idea
central, y el tema del siguiente documento.

---

← [Índice](00-indice.ipynb) · Siguiente: [Dos formas de saber](02-dos-formas-de-saber.ipynb) →
